# 如何搭建一个 `Module`

In [10]:
import torch 
import torch.nn as nn

class TestModule(nn.Module):
    def __init__(self):
        super().__init__()
        pass
    
    def forward(self):
        pass

+ 继承 nn.Module 的核心机制体现在以下四个方面：

    + 参数自动注册与追踪：在 `__init__` 中被赋值为 `nn.Module`(如 `nn.Linear` 或 `nn.Parameter` 的属性)，会被自动记录到内部字典（`_modules` 和 `_parameters`）中。这使得调用 `model.parameters()` 能直接返回整个计算图中所有可学习参数的迭代器，直接喂给优化器。

    + `__call__` 劫持与 Hook 机制：`nn.Module` 实现了 Python 的 `__call__` 魔法方法。执行 `model(x)` 时，底层会先触发所有注册的 `forward_pre_hooks`，然后才调用你重写的 `forward(x)` 方法，最后触发 `forward_hooks`。因此，规范做法是调用 `model(x)` 而非 `model.forward(x)`。

    + 递归的设备与精度管理：调用 `model.to('cuda')` 或 `model.half()` 时，父类会遍历整棵模块树，递归地将所有内部参数和缓冲区（buffers）迁移至目标计算设备或转换数据类型。

    + 运行上下文切换：通过 `model.train()` 和 `model.eval()` 递归修改内部的 `self.training` 布尔标志。这决定了诸如 `Dropout` 或 `BatchNorm` 等行为依赖于运行阶段的算子的正向传播逻辑。

+ `super().__init__()`: 调用当前类的父类的构造函数 `__init__()`. 这里注意 `super()` 有个括号

# 单头注意力模块

In [11]:
import torch
import torch.nn as nn
import math

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model):
        """
        初始化单头注意力机制
        :param d_model: 输入向量的维度 (例如 512)
        """
        super(SingleHeadAttention, self).__init__()
        
        self.d_model = d_model
        
        # Q, K, V 的独立线性映射层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # 最后的输出线性映射层
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        计算缩放点积注意力
        """
        # Q 和 K 的点积，再除以 sqrt(d_model)
        # Q shape: (batch_size, seq_len, d_model)
        # K.transpose shape: (batch_size, d_model, seq_len)
        # scores shape: (batch_size, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        
        # 如果有 mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # 计算注意力权重
        # attention_weights shape: (batch_size, seq_len, seq_len)
        attention_weights = torch.softmax(scores, dim=-1)
        
        # 权重与 V 相乘
        # output shape: (batch_size, seq_len, d_model)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

    def forward(self, query, key, value, mask=None):
        # 1. 线性映射 (去除了多头的拆分逻辑)
        # 最终 shape 保持为: (batch_size, seq_len, d_model)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. 计算 Scaled Dot-Product Attention
        # x shape: (batch_size, seq_len, d_model)
        x, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # 3. 最后的线性映射层 (去除了多头的拼接逻辑)
        # shape: (batch_size, seq_len, d_model)
        output = self.W_o(x)
        
        return output




In [12]:
if __name__ == "__main__":
    # --- 超参数设置 ---
    batch_size = 2      # 一次处理 2 个句子
    seq_len = 5         # 每个句子有 5 个词 (Token)
    d_model = 512       # 词向量的维度是 512
    
    # 打印一些提示信息
    print(f"初始化参数: batch_size={batch_size}, seq_len={seq_len}, d_model={d_model}")
    
    # --- 1. 构造输入数据 ---
    # X 的 shape 是 (batch_size, seq_len, d_model)
    X = torch.randn(batch_size, seq_len, d_model)
    print(f"输入张量 X 的形状: {X.shape}")
    
    # --- 2. 实例化单头注意力模块 ---
    attention = SingleHeadAttention(d_model=d_model)
    
    # --- 3. 前向传播计算 ---
    output = attention(query=X, key=X, value=X, mask=None)
    
    # --- 4. 检查输出 ---
    print(f"输出张量的形状: {output.shape}")
    
    # 验证输入和输出的形状是否一致
    assert output.shape == X.shape, "错误：输出形状与输入形状不一致！"
    print("测试通过！输入和输出的形状完全一致。")

初始化参数: batch_size=2, seq_len=5, d_model=512
输入张量 X 的形状: torch.Size([2, 5, 512])
输出张量的形状: torch.Size([2, 5, 512])
测试通过！输入和输出的形状完全一致。


+ 这里的输入的 query, key, value 的 shape 为 `(batch_size, seq_len, d_model)`, 即  batch 的大小, 序列的长度, 特征的维度.

+ `nn.Linear()` 构造全连接层, 只对最后一个维度进行线下映射. `nn.Linear(m, n)` 会构造出一个 `(m, n)` 的矩阵 $W$ 和一个 `(n,1)` 的向量 $b$, 调用这个模块会进行线性映射 $\mathbf{y} = W\mathbf{x}^T + \mathbf{b}$. 在我们的这个单头注意力模块, 数据流经 `W_q`, `W_k`, `W_v` 后 shape 仍然为 `(batch_size, seq_len, d_model)`

+ `transpose` 的作用是交换 Tensor 的两个维度. `K.transpose(-2, -1)` 会交换第 `-2` 个维度和第 `-1` 个维度, 所以 K 的 shape 变为 `(batch_size, d_mdoel, seq_len)`. 

+ `torch.matmul` 执行**批量矩阵乘法 (Batched Matrix Multiplication)**。它具有广播（Broadcasting）机制，会自动保留前面的 Batch 维度，仅对最后两个维度执行标准的线性代数矩阵乘法. 这里乘法的结果是 scores, 其 shape 为`(batch_size, seq_len, seq_len)`, 同样的, `attention_weights` 的 shape 也是 `(batch_size, seq_len, seq_len)`. 

+ `/ math.sqrt(self.d_model)` 是为了缩放控制方差, 防止 Softmax 函数梯度消失. 假设 `Q` 和 `K` 里面的元素都是独立的随机变量，且满足均值为 0，方差为 1。当它们进行点积操作 $q \cdot k = \sum_{i=1}^{d_{model}} q_i k_i$ 时，累加了 $d_{model}$. 根据统计学原理，这个点积结果的均值依然是 0，但方差会被放大到 $d_{model}$. 当特征维度 $d_{model}$ 较大（如 512）时，点积结果的绝对值会非常大. 将这些巨大的数值直接送入 Softmax 函数，会使得输出的概率分布极其陡峭（最大值趋近于 1，其余趋近于 0）。在这些极端区域，Softmax 的导数极小，导致反向传播时梯度几乎为 0，模型参数无法有效更新. 将点积结果除以 $\sqrt{d_{model}}$，恰好可以将其方差重新缩放回 1，保证模型在训练初期的稳定性。

+ `torch.softmax()`: 对于输入向量 $z$ 中的第 $i$ 个元素 $z_i$，其 Softmax 输出计算如下：
  $$Softmax(z_i) = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$$

+ `output = torch.matmul(attention_weights, V)` 得到的结果 shape 是 `(batch_size, seq_len, d_model)`, 这个结果是将所有的 value 根据之前算得的注意力权重 `attention_weights` 进行线性加权得到的. 

+ 掩码操作通过作用到 scores 实现, 注意写法
    ```python
    scores = scores.masked_fill(mask == 0, -1e9)
    ```
    这里的 `mask` 含义为 0 代表被遮挡的 token, 然后通过赋值负无穷来使得 softmax 后的结果接近 0. (注意不能在 softmax 后的结果做 mask, 否则得到的分数不满足求和为 1)


# MHA


In [13]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        初始化多头注意力机制
        :param d_model: 输入向量的维度 (例如 512)
        :param num_heads: 头的数量 (例如 8)
        """
        super(MultiHeadAttention, self).__init__()
        
        # 确保 d_model 可以被 num_heads 整除
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度 (例如 512 / 8 = 64)
        
        # 原论文中，Q, K, V 都有各自独立的线性映射层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # 最后的输出线性映射层
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        计算缩放点积注意力
        """
        # Q 和 K 的点积，再除以 sqrt(d_k)
        # Q shape: (batch_size, num_heads, seq_len, d_k)
        # K.transpose shape: (batch_size, num_heads, d_k, seq_len)
        # scores shape: (batch_size, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 如果有 mask (例如在 Decoder 中防止看到未来的词，或者 padding mask)
        if mask is not None:
            # 将 mask 中为 0 的位置替换为一个极小的数 (如 -1e9)，这样 softmax 后权重趋近于 0
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # 计算注意力权重
        attention_weights = torch.softmax(scores, dim=-1)
        
        # 权重与 V 相乘
        # output shape: (batch_size, num_heads, seq_len, d_k)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # 1. 线性映射并划分成 num_heads 个头
        # view 的操作将 d_model 拆分成 (num_heads, d_k)
        # transpose(1, 2) 是为了把 num_heads 移到前面，方便后续的矩阵乘法
        # 最终 shape: (batch_size, num_heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算 Scaled Dot-Product Attention
        # x shape: (batch_size, num_heads, seq_len, d_k)
        x, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # 3. 拼接所有的头 (Concat)
        # 把 num_heads 移回原来的位置，并将 num_heads 和 d_k 重新合并成 d_model
        # contiguous() 是因为 transpose 破坏了内存连续性，view 需要连续的内存
        # shape 变化: (batch, heads, seq, d_k) -> (batch, seq, heads, d_k) -> (batch, seq, d_model)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 4. 最后的线性映射层
        # shape: (batch_size, seq_len, d_model)
        output = self.W_o(x)
        
        return output

In [14]:
if __name__ == "__main__":
    # --- 超参数设置 ---
    batch_size = 2      # 一次处理 2 个句子
    seq_len = 5         # 每个句子有 5 个词 (Token)
    d_model = 512       # 词向量的维度是 512
    num_heads = 8       # 拆分成 8 个头
    
    # 打印一些提示信息
    print(f"初始化参数: batch_size={batch_size}, seq_len={seq_len}, d_model={d_model}, num_heads={num_heads}")
    
    # --- 1. 构造输入数据 ---
    # 在自注意力机制中，Q, K, V 通常来自于同一个输入 X
    # X 的 shape 是 (batch_size, seq_len, d_model)
    X = torch.randn(batch_size, seq_len, d_model)
    print(f"输入张量 X 的形状: {X.shape}")
    
    # --- 2. 实例化 MHA 模块 ---
    mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
    
    # --- 3. 前向传播计算 ---
    # 将 X 作为 Q, K, V 同时传入，这就是 "Self-Attention (自注意力)" 的名字由来
    output = mha(query=X, key=X, value=X, mask=None)
    
    # --- 4. 检查输出 ---
    print(f"输出张量的形状: {output.shape}")
    
    # 验证输入和输出的形状是否一致 (MHA 保持序列长度和维度不变)
    assert output.shape == X.shape, "错误：输出形状与输入形状不一致！"
    print("测试通过！输入和输出的形状完全一致。")

初始化参数: batch_size=2, seq_len=5, d_model=512, num_heads=8
输入张量 X 的形状: torch.Size([2, 5, 512])
输出张量的形状: torch.Size([2, 5, 512])
测试通过！输入和输出的形状完全一致。


+ MHA 无非是在单头自注意力的基础上, 加上了一个 head 维度, 然后把这些维度 concat 到了一起.